In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv('bersatu_finetune.csv')

# pilih kolom
df = df[['clean_textdisplay', 'label']].copy()

# hapus data kosong
df = df.dropna()

# rapihin label
df['label'] = (
    df['label']
    .astype(str)
    .str.lower()
    .str.strip()
)


In [3]:
# mapping label indonesia
label_mapping = {
    'negatif': 0,
    'netral': 1,
    'positif': 2
}

df['label'] = df['label'].map(label_mapping)

# cek hasil
print(df['label'].value_counts())
print(df.head())

label
1    800
0    800
2    800
Name: count, dtype: int64
                                   clean_textdisplay  label
0  ada ga ada mobil listrik pabrikan nikel terus ...      1
1  akhirnya persaingan negara china dengan electr...      1
2  baterai electric vehicle di recycle menjadi bl...      1
3  bcara baterai mobil bensin solar pun ada bater...      1
4  betul dokter mobil yg ceritakan ku pake mobil ...      1


In [4]:
df = df.dropna(subset=['label'])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [5]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: fb91cda6-5fbf-49d7-a287-8b164d2be64d)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: edb7153c-2b4d-474c-9385-5ee54eb92fdf)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 620a4bb9-95c1-487d-8348-e4d9ee64f19b)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json
Retrying in 4s [Retry 3/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: e0f1a39

In [6]:
def tokenize_function(example):
    return tokenizer(
        example['clean_textdisplay'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

In [7]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 036b858e-4c97-4aaf-95d8-82477692ba6a)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: d2b10e76-37bb-4aec-97b0-def3db758ebc)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 531701fb-9607-4ca6-a0a8-ce82ebe441df)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: e16b2cd6-1fdb-4448-b9b9-8b703a606648)

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc
    }

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    logging_steps=10
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [11]:
trainer.train()

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.866700,0.803237,0.666667


TrainOutput(global_step=120, training_loss=0.8844126502672831, metrics={'train_runtime': 68.1182, 'train_samples_per_second': 28.186, 'train_steps_per_second': 1.762, 'total_flos': 126294440509440.0, 'train_loss': 0.8844126502672831, 'epoch': 1.0})

In [12]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nAccuracy:")
print(accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Accuracy:
0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.57      0.60       160
           1       0.72      0.71      0.71       160
           2       0.65      0.72      0.68       160

    accuracy                           0.67       480
   macro avg       0.67      0.67      0.67       480
weighted avg       0.67      0.67      0.67       480


Confusion Matrix:
[[ 92  29  39]
 [ 24 113  23]
 [ 29  16 115]]


In [13]:
df['clean_textdisplay'].duplicated().sum()

np.int64(42)